In [1]:
import cv2
import numpy as np
import time
import threading
import os

# importing deep learning libraries
try:
    from tensorflow.keras.models import Sequential, load_model
    from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
    from tensorflow.keras.utils import to_categorical
    from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
    KERAS_AVAILABLE = True
except ImportError:
    try:
        from keras.models import Sequential, load_model
        from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
        from keras.utils import to_categorical
        from keras.callbacks import ModelCheckpoint, EarlyStopping
        KERAS_AVAILABLE = True
    except ImportError:
        KERAS_AVAILABLE = False
        print("[WARNING] TensorFlow/Keras not found. Using classical CV fallback.")

In [2]:
# ─── BEEP FUNCTION ────────────────────────────────────────────────────────────

def beep_alert():
    """Play a beep sound in a separate thread (non-blocking)."""
    def _beep():
        # Try multiple methods for cross-platform beep
        try:
            import winsound
            winsound.Beep(1000, 300)  # Windows
        except ImportError:
            try:
                import subprocess
                subprocess.run(['beep', '-f', '1000', '-l', '300'], 
                             capture_output=True, timeout=1)
            except Exception:
                # Fallback: terminal bell
                print('\a', end='', flush=True)
    
    t = threading.Thread(target=_beep, daemon=True)
    t.start()

In [3]:
# ─── CNN MODEL ────────────────────────────────────────────────────────────────

def build_cnn_model(input_shape=(24, 24, 1)):
    """
    Build a CNN model for eye open/closed classification.
    
    Architecture:
    Conv2D -> BatchNorm -> MaxPool -> Dropout
    Conv2D -> BatchNorm -> MaxPool -> Dropout
    Flatten -> Dense -> Dense(output)
    """
    model = Sequential([
        # Block 1
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),

        # Block 2
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),

        # Block 3
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.4),

        # Classifier
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(2, activation='softmax')  # 0=Closed, 1=Open
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [4]:
def train_model_on_dataset(data_dir='dataset', model_save_path='eye_cnn_model.h5'):
    """
    Train CNN on eye dataset.
    
    Expected dataset structure:
    dataset/
      open/   <- images of open eyes
      closed/ <- images of closed eyes
    """
    print("\n[INFO] Loading dataset...")
    images, labels = [], []
    IMG_SIZE = 24

    for label_idx, category in enumerate(['closed', 'open']):
        folder = os.path.join(data_dir, category)
        if not os.path.exists(folder):
            print(f"[ERROR] Folder not found: {folder}")
            continue
        
        for fname in os.listdir(folder):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                img_path = os.path.join(folder, fname)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    continue
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = img / 255.0
                images.append(img)
                labels.append(label_idx)
    if len(images) == 0:
        print("[ERROR] No images found in dataset.")
        return None

    X = np.array(images).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
    y = to_categorical(np.array(labels), 2)

    # Train/Test split
    split = int(0.8 * len(X))
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    print(f"[INFO] Training samples: {len(X_train)}, Test samples: {len(X_test)}")

    model = build_cnn_model(input_shape=(IMG_SIZE, IMG_SIZE, 1))
    model.summary()

    callbacks = [
        ModelCheckpoint(model_save_path, save_best_only=True, monitor='val_accuracy'),
        EarlyStopping(patience=5, restore_best_weights=True)
    ]

    print("\n[INFO] Training started...")
    model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=30,
        batch_size=32,
        callbacks=callbacks
    )

    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"\n[RESULT] Test Accuracy: {acc*100:.2f}%")
    return model


In [5]:
# ─── CLASSICAL EAR FALLBACK ──────────────────────────────────────────────────

def compute_EAR(eye_points):
    """
    Eye Aspect Ratio (EAR) - classical fallback method.
    EAR = (||p2-p6|| + ||p3-p5||) / (2 * ||p1-p4||)
    When eye closes, EAR drops below threshold.
    """
    def dist(a, b):
        return np.linalg.norm(np.array(a) - np.array(b))
    
    A = dist(eye_points[1], eye_points[5])
    B = dist(eye_points[2], eye_points[4])
    C = dist(eye_points[0], eye_points[3])
    
    if C == 0:
        return 0.0
    return (A + B) / (2.0 * C)

In [6]:
# ─── HAAR CASCADE DETECTION ──────────────────────────────────────────────────

def load_cascades():
    """Load OpenCV Haar Cascade classifiers."""
    face_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
    )
    eye_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + 'haarcascade_eye.xml'
    )
    
    # Try loading eye-glasses cascade as fallback
    left_eye_cascade  = cv2.CascadeClassifier(
        cv2.data.haarcascades + 'haarcascade_lefteye_2splits.xml'
    )
    right_eye_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + 'haarcascade_righteye_2splits.xml'
    )
    
    return face_cascade, eye_cascade, left_eye_cascade, right_eye_cascade

In [7]:
def preprocess_eye(eye_roi, size=24):
    """Preprocess eye ROI for CNN input."""
    gray = cv2.cvtColor(eye_roi, cv2.COLOR_BGR2GRAY) if len(eye_roi.shape) == 3 else eye_roi
    resized = cv2.resize(gray, (size, size))
    normalized = resized / 255.0
    return normalized.reshape(1, size, size, 1)


In [ ]:
# ─── MAIN DETECTION LOOP ─────────────────────────────────────────────────────

def run_detection(model=None, model_path='eye_cnn_model.h5'):
    """
    Main real-time detection loop.
    
    - Uses CNN if available and trained
    - Falls back to Haar Cascade + pixel analysis
    """
    print("\n" + "="*55)
    print("  EYE BLINK DETECTION SYSTEM - Starting...")
    print("="*55)
    print("  Press 'Q' to quit")
    print("  Press 'S' to save screenshot")
    print("="*55 + "\n")

    # Load model
    use_cnn = False
    if KERAS_AVAILABLE and model is None and os.path.exists(model_path):
        print(f"[INFO] Loading saved model: {model_path}")
        model = load_model(model_path)
        use_cnn = True
    elif model is not None:
        use_cnn = True

    if use_cnn:
        print("[INFO] Mode: CNN-based detection")
    else:
        print("[INFO] Mode: Classical CV detection (Haar Cascade)")

    # Load cascades
    face_cascade, eye_cascade, left_eye_cas, right_eye_cas = load_cascades()

    # State variables
    cap = cv2.VideoCapture(0,cv2.CAP_DSHOW)  # Use DirectShow backend for better performance on Windows
    if not cap.isOpened():
        print("[ERROR] Camera not found! Check webcam connection.")
        return

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    EYE_CLOSED_THRESHOLD = 0.9   # CNN confidence threshold
    CLOSED_FRAMES_ALERT  = 10    # Frames closed before alert
    PIXEL_DARK_THRESHOLD = 80    # Classical: pixel darkness threshold
    EAR_THRESHOLD        = 0.21  # EAR threshold for classical method

    closed_frame_count = 0
    beep_cooldown      = 0
    frame_count        = 0
    status             = "DETECTING..."
    status_color       = (200, 200, 200)

    # Font settings
    FONT      = cv2.FONT_HERSHEY_DUPLEX
    FONT_BOLD = cv2.FONT_HERSHEY_TRIPLEX

    print("[INFO] Camera opened. Detection running...\n")

    while True:
        ret, frame = cap.read()
        if not ret:
            print("[ERROR] Frame read failed.")
            break

        frame_count += 1
        display = frame.copy()
        gray    = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray    = cv2.equalizeHist(gray)

        # ── Face Detection ──────────────────────────
        faces = face_cascade.detectMultiScale(
            gray, scaleFactor=1.1, minNeighbors=5, minSize=(80, 80)
        )

        eye_status_list = []  # Collect status of each detected eye

        if len(faces) > 0:
            fx, fy, fw, fh = faces[0]  # Process largest face
            cv2.rectangle(display, (fx, fy), (fx+fw, fy+fh), (70, 130, 180), 2)
            cv2.putText(display, "Chehra", (fx, fy-8), FONT, 0.5, (70, 130, 180), 1)

            face_roi_gray  = gray[fy:fy+fh, fx:fx+fw]
            face_roi_color = frame[fy:fy+fh, fx:fx+fw]

            # ── Eye Detection inside face ROI ────────
            eyes = eye_cascade.detectMultiScale(
                face_roi_gray, scaleFactor=1.05, minNeighbors=4, minSize=(20, 20)
            )

            for (ex, ey, ew, eh) in eyes[:2]:  # Max 2 eyes
                # Absolute coords for display
                abs_ex = fx + ex
                abs_ey = fy + ey

                eye_roi_color = face_roi_color[ey:ey+eh, ex:ex+ew]
                eye_roi_gray  = face_roi_gray[ey:ey+eh, ex:ex+ew]

                if eye_roi_gray.size == 0:
                    continue

                # ── Classification ────────────────────
                if use_cnn:
                    eye_input = preprocess_eye(eye_roi_color)
                    pred = model.predict(eye_input, verbose=0)[0]
                    # pred[0] = closed probability, pred[1] = open probability
                    is_open = pred[1] > EYE_CLOSED_THRESHOLD
                    conf    = pred[1] if is_open else pred[0]
                    method_label = f"CNN {conf*100:.0f}%"
                else:
                    # Classical: average pixel intensity in pupil region
                    h, w     = eye_roi_gray.shape
                    center   = eye_roi_gray[h//4:3*h//4, w//4:3*w//4]
                    avg_pix  = np.mean(center)
                    is_open  = avg_pix > PIXEL_DARK_THRESHOLD
                    method_label = f"CV {avg_pix:.0f}"

                eye_status_list.append(is_open)

                # Draw eye box
                color = (0, 220, 100) if is_open else (0, 60, 220)
                cv2.rectangle(display, (abs_ex, abs_ey), 
                              (abs_ex+ew, abs_ey+eh), color, 2)
                cv2.putText(display, method_label, 
                            (abs_ex, abs_ey - 5), FONT, 0.38, color, 1)

        # ── Overall Status ───────────────────────────
        if len(eye_status_list) > 0:
            # Eyes closed if majority are closed
            all_open = any(eye_status_list)

            if all_open:
                status       = "Aankhe Khuli Hain  :)"
                status_color = (0,220,80)
                
                closed_frame_count = 0
            else:
                closed_frame_count += 1
                status       = "Aankhe Band Hain  :("
                status_color = (0, 80, 255)
               

                # BEEP alert after threshold
                if closed_frame_count >= CLOSED_FRAMES_ALERT:
                    if beep_cooldown <= 0:
                        beep_alert()
                        beep_cooldown = 15  # frames cooldown
                        print(f"[ALERT] * AANKHE BAND HAIN! * (Frame {frame_count})")
        else:
            status       = "Chehra Nahi Mila..."
            status_color = (180, 180, 40)
            closed_frame_count = 0

        if beep_cooldown > 0:
            beep_cooldown -= 1

        # ── HUD Overlay ──────────────────────────────
        # Semi-transparent top bar
        overlay = display.copy()
        cv2.rectangle(overlay, (0, 0), (640, 55), (15, 15, 15), -1)
        cv2.addWeighted(overlay, 0.75, display, 0.25, 0, display)

        # Title
        cv2.putText(display, "EYE BLINK DETECTOR", (10, 22),
                    FONT_BOLD, 0.65, (255, 220, 50), 1)

        # Mode indicator
        mode_txt = "CNN Mode" if use_cnn else "CV Mode"
        cv2.putText(display, mode_txt, (490, 22), FONT, 0.45, (150, 150, 150), 1)

        # Status bar at bottom
        overlay2 = display.copy()
        cv2.rectangle(overlay2, (0, 420), (640, 480), (15, 15, 15), -1)
        cv2.addWeighted(overlay2, 0.8, display, 0.2, 0, display)

        cv2.putText(display, status, (15, 460),
                    FONT_BOLD, 0.8, status_color, 1)

        # Frame counter
        cv2.putText(display, f"Frame: {frame_count}", (530, 460),
                    FONT, 0.4, (100, 100, 100), 1)

        # ALERT flash when beeping
        if closed_frame_count >= CLOSED_FRAMES_ALERT:
            flash_overlay = display.copy()
            cv2.rectangle(flash_overlay, (0, 0), (640, 480), (0, 0, 180), 3)
            alpha = 0.3 + 0.3 * np.sin(frame_count * 0.3)
            cv2.addWeighted(flash_overlay, alpha, display, 1-alpha, 0, display)

            cv2.putText(display, "! ALERT !", (245, 240),
                        FONT_BOLD, 1.2, (0, 80, 255), 2)

        # Show frame
        cv2.imshow("Eye Blink Detection System - Press Q to Quit", display)

        # Key controls
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == ord('Q'):
            print("\n[INFO] Quitting...")
            break
        elif key == ord('s') or key == ord('S'):
            fname = f"screenshot_{frame_count}.png"
            cv2.imwrite(fname, display)
            print(f"[INFO] Screenshot saved: {fname}")

    cap.release()
    cv2.destroyAllWindows()
    print("[INFO] Detection stopped.")


In [19]:
# ─── ENTRY POINT ──────────────────────────────────────────────────────────────

if __name__ == "_main_":
    import sys

    print("""
╔══════════════════════════════════════════════╗
║    EYE BLINK DETECTION SYSTEM                ║
║    CNN + OpenCV | ML Final Project           ║
╠══════════════════════════════════════════════╣
║  1. Sirf detection chalao (no training)      ║
║  2. Model train karo (dataset chahiye)       ║
╚══════════════════════════════════════════════╝
    """)

    if len(sys.argv) > 1 and sys.argv[1] == 'train':
        # Training mode
        if KERAS_AVAILABLE:
            data_dir = sys.argv[2] if len(sys.argv) > 2 else 'dataset'
            print(f"[INFO] Training on dataset: {data_dir}")
            trained_model = train_model_on_dataset(data_dir=data_dir)
            if trained_model:
                print("\n[INFO] Training complete! Ab detection start kar raha hai...")
                run_detection(model=trained_model)
        else:
            print("[ERROR] TensorFlow not installed. pip install tensorflow")
    else:
        # Detection mode
        run_detection()

In [20]:
run_detection()


  EYE BLINK DETECTION SYSTEM - Starting...
  Press 'Q' to quit
  Press 'S' to save screenshot

[INFO] Mode: Classical CV detection (Haar Cascade)
[INFO] Camera opened. Detection running...


[INFO] Quitting...
[INFO] Detection stopped.
